# Stage 01: Problem Framing & Scoping

In [7]:
import os, pathlib, textwrap

# Locate project repository root path
REPO_PATH = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

# Step 1: Create standard project directory structure
for d in ["data/raw", "data/processed", "src", "notebooks", "docs", "reports", "model"]:
    (REPO_PATH / d).mkdir(parents=True, exist_ok=True)

In [9]:
# Step 2: Write project/README.md
readme_tmpl = textwrap.dedent("""
# S&P 500 Historical Asset Return & Volatility Prediction

## Problem Statement
Quantitative portfolio managers need reliable predictive models to understand short-term asset return dynamics and price volatility. Evaluating asset behavior over historical market regimes helps quantitative analysts design robust allocation strategies and quantify downside risks.
This project utilizes a static 5-year historical daily dataset of the S&P 500 index (2020–2025) stored in `data/raw/sp500_daily_history.csv`. The goal is to build a quantitative modeling pipeline that engineers technical features and fits predictive models to forecast next-day index returns and volatility.

## Stakeholder & User
* **Primary Stakeholder:** Portfolio Manager (PM) and Investment Committee.
* **End User:** Quantitative Research Analyst evaluating strategy feasibility.
* **Workflow Context:** Batch model evaluation based on fixed historical backtesting data to inform risk management policy.

## Useful Answer & Decision
* **Answer Type:** Predictive (Quantitative Point Forecast & Volatility Bands).
* **Primary Metric:** Root Mean Squared Error (RMSE) for return predictions and $R^2$ for goodness of fit.
* **Artifact Delivered:** Predictive model pipeline, diagnostic report, and feature importance analysis based on historical dataset.

## Assumptions & Constraints
* **Data Source:** Static historical daily OHLCV dataset (`sp500_daily_history.csv`). No live API dependency required during model development.
* **Scope:** Daily time-frequency sampling over a fixed 5-year window.
* **Constraint:** Model evaluation must account for transaction frictions and regime shifts within the historical timeframe.

## Known Unknowns / Risks
* **Non-stationarity:** Financial time series non-stationarity causing baseline OLS assumption violations.
* **Overfitting Risk:** Spurious correlations among engineered technical features (mitigated via out-of-sample split and regularization).

## Lifecycle Mapping
- Scope problem & set static dataset → Problem Framing & Scoping (Stage 01) → Stakeholder Memo & README
- Feature engineering on historical CSV → Data Prep & Feature Engineering (Stage 02-09) → Pipeline Scripts in `src/`
- Build OLS baseline & check diagnostics → Modeling (Stage 10a) → OLS Diagnostic Plots & Metric Tables
- Evaluate time series/GARCH volatility → Modeling (Stage 10b) → Time Series Models & Final Evaluation

## Repo Plan
- `data/`: `raw/sp500_daily_history.csv` and processed feature matrices.
- `src/`: Modular Python scripts for feature creation and modeling.
- `notebooks/`: Exploratory Data Analysis and regression diagnostics.
- `docs/`: Stakeholder memos and project documentation.
""").strip()

(REPO_PATH / "README.md").write_text(readme_tmpl, encoding="utf-8")
print(f"README saved to: {(REPO_PATH / 'README.md').resolve()}")

README saved to: /Users/cloudnine_7/bootcamp_elrie_lin/project/README.md


In [10]:
# Step 3: Write project/docs/stakeholder_memo.md
memo_text = textwrap.dedent("""
# Stakeholder Brief — S&P 500 Historical Return & Volatility Model
**Audience:** Investment Committee / Portfolio Manager | **Cadence:** Batch Historical Analysis  
**Decision Supported:** Quantitative Model Strategy Evaluation & Asset Allocation Research  

## Context
Understanding historical asset volatility and return drivers is essential for developing quantitative trading strategies. Using a fixed historical dataset allows for rigorous model diagnostics and hypothesis testing without live API variability.

## What You'll Receive
- **Baseline Forecasting Models:** OLS and time series models predicting daily index return movements.
- **Diagnostic Reports:** Statistical checks on regression assumptions (linearity, homoscedasticity, residual independence).
- **Feature Importance Matrix:** Quantitative summary of key predictors (lagged returns, rolling volatility, moving averages).

## Assumptions & Constraints
- Dataset is fixed to 2020–2025 S&P 500 daily trading data (`sp500_daily_history.csv`).
- Model evaluations will utilize an 80/20 train-test historical split to prevent look-ahead bias.
""").strip()

(REPO_PATH / "docs" / "stakeholder_memo.md").write_text(memo_text, encoding="utf-8")
print(f"Memo saved to: {(REPO_PATH / 'docs' / 'stakeholder_memo.md').resolve()}")

Memo saved to: /Users/cloudnine_7/bootcamp_elrie_lin/project/docs/stakeholder_memo.md


# Stage 02: Tooling Setup

In [13]:
import os
import pathlib
import textwrap

# Locate the project root folder
REPO_PATH = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

# 1. Create standard project scaffolding folders
directories = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src",
    "docs",
    "reports",
    "model"
]
for d in directories:
    (REPO_PATH / d).mkdir(parents=True, exist_ok=True)
print("Project folder structure verified.")

Project folder structure verified.


In [14]:
# 2. Create .env.example and local .env file
env_example_content = textwrap.dedent("""
API_KEY=dummy_key_123
DATA_DIR=./data
""").strip()

(REPO_PATH / ".env.example").write_text(env_example_content, encoding="utf-8")

if not (REPO_PATH / ".env").exists():
    (REPO_PATH / ".env").write_text(env_example_content, encoding="utf-8")
    print(".env file initialized from .env.example.")

In [15]:
# 3. Create project/src/config.py for centralized path management
config_content = textwrap.dedent("""
import os
from pathlib import Path
from dotenv import load_dotenv

def load_env():
    \"\"\"Load environment variables from .env file.\"\"\"
    load_dotenv()

def get_key(name: str, default=None):
    \"\"\"Retrieve environment variable by key name.\"\"\"
    return os.getenv(name, default)

# Load environment on import
load_env()

# Centralized paths
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / get_key("DATA_DIR", "./data")
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
""").strip()

(REPO_PATH / "src" / "config.py").write_text(config_content, encoding="utf-8")
print("src/config.py created successfully.")

src/config.py created successfully.


In [16]:
# 4. Verify config.py in the notebook
import sys
sys.path.append(str(REPO_PATH / "src"))
import config

print("\n--- Project Environment Verification ---")
print("Project Root:", config.PROJECT_ROOT)
print("Raw Data Directory:", config.RAW_DATA_DIR)
print("API_KEY Loaded:", config.get_key("API_KEY") is not None)


--- Project Environment Verification ---
Project Root: /Users/cloudnine_7/bootcamp_elrie_lin/project
Raw Data Directory: /Users/cloudnine_7/bootcamp_elrie_lin/project/data/raw
API_KEY Loaded: True


# Stage 03: Python Fundamentals

In [24]:
import pathlib
import textwrap

# Locate project root directory
REPO_PATH = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

# Ensure src directory exists
(REPO_PATH / "src").mkdir(parents=True, exist_ok=True)

# Write project/src/utils.py
utils_code = textwrap.dedent("""
import numpy as np
import pandas as pd

def get_summary_stats(df: pd.DataFrame, numeric_cols: list = None) -> pd.DataFrame:
    \"\"\"
    Generates key statistical metrics for specified numeric features.
    \"\"\"
    if numeric_cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    return df[numeric_cols].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
""").strip()

(REPO_PATH / "src" / "utils.py").write_text(utils_code, encoding="utf-8")
print("Successfully written project/src/utils.py!")

Successfully written project/src/utils.py!


# Stage 04: Data Acquisition and Ingestion

In [26]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..') # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) # so `from src....` imports work
print('working from:', ROOT.name)

working from: project


In [27]:
# Setup, Directories & Dependencies
import datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# Define raw data path
RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Load environment variables
load_dotenv(dotenv_path=ROOT / '.env')
print('API Key Loaded:', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

API Key Loaded: True


In [28]:
# Helper Functions for Pipeline
def get_timestamp():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_raw_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k, v in meta.items()])
    filename = f"{prefix}_{mid}_{get_timestamp()}.csv"
    path = RAW_DIR / filename
    df.to_csv(path, index=False)
    print(f"Successfully saved raw data -> {path}")
    return path

def validate_dataframe(df: pd.DataFrame, required_cols: list):
    missing = [c for c in required_cols if c not in df.columns]
    status = {
        'missing_columns': missing,
        'shape': df.shape,
        'na_count': int(df.isna().sum().sum()),
        'is_valid': len(missing) == 0 and not df.empty
    }
    return status

In [31]:
# --- Step 1: Ingest S&P 500 Historical Market Data (^GSPC) ---
SYMBOL = '^GSPC'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY')) and os.getenv('ALPHAVANTAGE_API_KEY') != 'your_api_key_here'

if USE_ALPHA:
    print(f"Fetching {SYMBOL} price data via Alpha Vantage API...")
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'TIME_SERIES_DAILY_ADJUSTED',
        'symbol': SYMBOL,
        'outputsize': 'full',
        'apikey': os.getenv('ALPHAVANTAGE_API_KEY')
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    js = resp.json()
    
    key = [k for k in js if 'Time Series' in k][0]
    df_api = pd.DataFrame(js[key]).T.reset_index()
    
    # Rename and select standardized columns
    df_api = df_api.rename(columns={
        'index': 'date',
        '1. open': 'open',
        '2. high': 'high',
        '3. low': 'low',
        '4. close': 'close',
        '5. adjusted close': 'adj_close',
        '6. volume': 'volume'
    })[['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']]
    
    source_label = 'alphavantage'

else:
    print(f"Alpha Vantage key unavailable/placeholder. Fetching {SYMBOL} via yfinance fallback...")
    import yfinance as yf
    
    # Fetch historical daily market data
    df_raw = yf.download(SYMBOL, period='5y', interval='1d', auto_adjust=False, multi_level_index=False)
    
    # Reset index to bring Date into columns
    df_api = df_raw.reset_index()
    
    # Flatten MultiIndex headers if present to prevent column shifting
    if isinstance(df_api.columns, pd.MultiIndex):
        df_api.columns = df_api.columns.get_level_values(0)
    
    # Normalize column names to lowercase snake_case
    df_api.columns = [str(col).strip().lower().replace(' ', '_') for col in df_api.columns]
    
    # Ensure fallback consistency between adj_close and close
    if 'adj_close' not in df_api.columns and 'close' in df_api.columns:
        df_api['adj_close'] = df_api['close']
        
    df_api = df_api[['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']]
    source_label = 'yfinance'

# Parse column data types
df_api['date'] = pd.to_datetime(df_api['date'])
numeric_cols = ['open', 'high', 'low', 'close', 'adj_close', 'volume']
for col in numeric_cols:
    df_api[col] = pd.to_numeric(df_api[col], errors='coerce')

# Sort chronologically by date
df_api = df_api.sort_values('date').reset_index(drop=True)

# Schema validation
v_api = validate_dataframe(df_api, ['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume'])
print("\nAPI Ingestion Validation Results:", v_api)
print("\nPreview of S&P 500 Market Data:")
print(df_api.head(3))

[*********************100%***********************]  1 of 1 completed

Alpha Vantage key unavailable/placeholder. Fetching ^GSPC via yfinance fallback...

API Ingestion Validation Results: {'missing_columns': [], 'shape': (1254, 7), 'na_count': 0, 'is_valid': True}

Preview of S&P 500 Market Data:
        date         open         high          low        close    adj_close  \
0 2021-08-25  4490.450195  4501.709961  4485.660156  4496.189941  4496.189941   
1 2021-08-26  4493.750000  4495.899902  4468.990234  4470.000000  4470.000000   
2 2021-08-27  4474.100098  4513.330078  4474.100098  4509.370117  4509.370117   

       volume  
0  3444700000  
1  3263980000  
2  3331200000  


In [32]:
# Save raw dataset with clean symbol tag
clean_symbol = SYMBOL.replace('^', '')
_ = save_raw_csv(df_api, prefix='api', source=source_label, symbol=clean_symbol)

Successfully saved raw data -> /Users/cloudnine_7/bootcamp_elrie_lin/project/data/raw/api_source-yfinance_symbol-GSPC_20260824-161525.csv


In [33]:
# --- Step 2: Scrape S&P 500 Constituents Metadata ---
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print(f"Scraping S&P 500 constituents table from {SCRAPE_URL}...")
resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.find('table', {'id': 'constituents'}) or soup.find('table', {'class': 'wikitable'})

# Extract raw tabular rows
rows = []
for tr in table.find_all('tr'):
    cols = [c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
    if cols:
        rows.append(cols)

header, *data = rows
df_raw_scrape = pd.DataFrame(data)

# Dynamic index matching to handle HTML DOM label variations
clean_headers = [str(h).strip().lower() for h in header]
sym_idx = next(i for i, h in enumerate(clean_headers) if 'symbol' in h or 'ticker' in h)
sec_idx = next(i for i, h in enumerate(clean_headers) if 'security' in h or 'company' in h)
sec_cat_idx = next(i for i, h in enumerate(clean_headers) if 'sector' in h)

df_scrape = pd.DataFrame({
    'symbol': df_raw_scrape[sym_idx],
    'security': df_raw_scrape[sec_idx],
    'sector': df_raw_scrape[sec_cat_idx]
})

# Schema validation
v_scrape = validate_dataframe(df_scrape, ['symbol', 'security', 'sector'])
print("\nWeb Scraping Validation Results:", v_scrape)
print("\nPreview of S&P 500 Constituents Data:")
print(df_scrape.head(3))



Scraping S&P 500 constituents table from https://en.wikipedia.org/wiki/List_of_S%26P_500_companies...

Web Scraping Validation Results: {'missing_columns': [], 'shape': (503, 3), 'na_count': 0, 'is_valid': True}

Preview of S&P 500 Constituents Data:
  symbol             security       sector
0    MMM                   3M  Industrials
1    AOS          A. O. Smith  Industrials
2    ABT  Abbott Laboratories  Health Care


In [34]:
# Save raw scraped dataset
_ = save_raw_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500constituents')

Successfully saved raw data -> /Users/cloudnine_7/bootcamp_elrie_lin/project/data/raw/scrape_site-wikipedia_table-sp500constituents_20260824-161946.csv


## Documentation & Pipeline Overview

### Data Sources & Ingestion Protocol
1. **Financial Market Time Series (API Ingestion)**:
   * **Source**: `yfinance` / Alpha Vantage (`^GSPC` ticker).
   * **Attributes**: `date`, `open`, `high`, `low`, `close`, `adj_close`, `volume`.
   * **Cleansing**: Resolved MultiIndex header misalignment, standardized snake_case naming, and explicitly cast date and numerical dtypes.
   * **Persistence**: Saved under `data/raw/api_source-yfinance_symbol-GSPC_YYYYMMDD-HHMMSS.csv`.

2. **Market Constituents Metadata (Web Scraping)**:
   * **Source**: Wikipedia (`List_of_S%26P_500_companies`).
   * **Attributes**: `symbol`, `security`, `sector`.
   * **Cleansing**: Parsed HTML DOM via `BeautifulSoup`, mapped column indices dynamically to resolve structural fragility.
   * **Persistence**: Saved under `data/raw/scrape_site-wikipedia_table-sp500constituents_YYYYMMDD-HHMMSS.csv`.

### Assumptions & Risks
* **Data Schema Resilience**: Dynamic column matching prevents pipeline failure in case of DOM changes.
* **API Rate Limits**: Built-in fallback mechanism handles missing API keys or request throttling seamlessly.
* **Security**: API keys are isolated within local `.env` and strictly excluded via `.gitignore`.